In [2]:
import os
import torch
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, 
                            classification_report, roc_auc_score)
from torch.utils.data._utils.collate import default_collate

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# 自定义collate函数处理PIL图像
def collate_fn(batch):
    # 将图像和标签分开
    images, targets = zip(*batch)
    # 返回图像列表和标签张量
    return list(images), torch.tensor(targets)

@torch.no_grad()
def zero_shot_classifier(model, processor, classnames, templates, device=None):
    """
    构建零样本分类器权重矩阵
    
    Args:
        model: CLIP模型
        processor: CLIP处理器
        classnames: 每个类别的名称列表的列表（每个类别可能有多个描述）
        templates: 文本提示模板列表
        device: 计算设备
    
    Returns:
        normalized_text_embeddings: 分类器权重矩阵 [embedding_dim, num_classes]
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    zeroshot_weights = []
    
    for classnames_for_class in classnames:
        embeddings_for_class = []
        for classname in classnames_for_class:
            # 使用模板创建提示词
            texts = [template.replace('CLASSNAME', classname) for template in templates]
            
            # 批量处理提示词以提高效率
            inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
            
            # 获取文本特征
            with torch.no_grad():
                text_features = model.get_text_features(**inputs)
                text_features = F.normalize(text_features, dim=-1)
            
            # 存储该类名的嵌入
            embeddings_for_class.append(text_features)
        
        # 平均每个类别中所有类名的嵌入
        class_embedding = torch.cat(embeddings_for_class, dim=0)
        class_embedding = class_embedding.mean(dim=0)
        class_embedding = F.normalize(class_embedding, dim=-1)
        
        zeroshot_weights.append(class_embedding)
    
    # 堆叠所有类别的嵌入
    zeroshot_weights = torch.stack(zeroshot_weights, dim=1)
    return zeroshot_weights

@torch.no_grad()
def run_zeroshot(model, processor, classifier, dataloader, device, metrics=['bacc', 'weighted_f1']):
    """
    执行零样本分类评估
    
    Args:
        model: CLIP模型
        processor: CLIP处理器
        classifier: 零样本分类器权重矩阵
        dataloader: 数据加载器
        device: 计算设备
        metrics: 要计算的指标列表
    
    Returns:
        results: 指标结果
        dump: 包含预测和标签的详细信息
    """
    acc_meter = AverageMeter()
    
    logits_all, targets_all, preds_all = [], [], []
    
    for batch_idx, (imgs, targets) in enumerate(tqdm(dataloader)):
        # 处理批量图像
        inputs = processor(images=imgs, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 获取图像特征
        image_features = model.get_image_features(**inputs)
        image_features = F.normalize(image_features, dim=-1)
        
        targets = torch.tensor(targets).to(device)
        batch_size = targets.size(0)
        
        # 计算相似度得分
        logits = 100.0 * (image_features @ classifier)
        preds = logits.argmax(dim=1)
        
        # 保存结果
        logits_all.append(logits.cpu().numpy())
        targets_all.append(targets.cpu().numpy())
        preds_all.append(preds.cpu().numpy())
        
        # 更新准确率
        acc_meter.update((preds == targets).float().mean().item(), n=batch_size)
    
    # 合并所有批次的结果
    targets_all = np.concatenate(targets_all, axis=0)
    logits_all = np.concatenate(logits_all, axis=0)
    probs_all = F.softmax(torch.from_numpy(logits_all) / 100.0, dim=1).numpy()
    preds_all = np.concatenate(preds_all, axis=0)
    
    # 计算各种指标
    bacc = balanced_accuracy_score(targets_all, preds_all)
    weighted_kappa = cohen_kappa_score(targets_all, preds_all, weights='quadratic')
    kappa = cohen_kappa_score(targets_all, preds_all)
    cls_rep = classification_report(targets_all, preds_all, output_dict=True, zero_division=0)
    acc = acc_meter.avg
    
    # 计算ROC AUC
    n_classes = probs_all.shape[1]
    if n_classes == 2:
        class_probs = probs_all[:,1]
        roc_kwargs = {}
    else:
        class_probs = probs_all
        roc_kwargs = {'multi_class': 'ovo', 'average': 'macro'}
    
    try:
        roc_auc = roc_auc_score(targets_all, class_probs, **roc_kwargs)
    except ValueError:
        roc_auc = np.nan
    
    # 收集所有计算的指标
    results = {'acc': acc, 
            'bacc': bacc, 
            'weighted_kappa': weighted_kappa,
            'kappa': kappa,
            'roc_auc': roc_auc,
            'weighted_f1': cls_rep['weighted avg']['f1-score']}
    
    # 只保留请求的指标
    results = {k: results[k] for k in metrics if k in results}
    
    # 准备详细输出
    dump = {
        'logits': logits_all,
        'targets': targets_all,
        'preds': preds_all,
    }
    
    return results, dump

def load_model_and_processor(model_path):
    """加载CLIP模型和处理器"""
    processor = CLIPProcessor.from_pretrained(model_path)
    model = CLIPModel.from_pretrained(model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, processor, device

def eval(model_path=None):
    # 加载模型和处理器
    print(f"Loading CLIP model from {model_path}...")
    model, processor, device = load_model_and_processor(model_path)
    
    # 设置数据路径
    root = Path('.').resolve()
    data_source = '/data/ljd/LLaVa/datasets/crc100k/CRC-VAL-HE-7K/'
    
    # 加载提示词文件
    prompt_file = '/home/hdc/ljd1/github/conch/prompts/crc100k_prompts_all_per_class.json'
    
    # 创建数据加载器 - 使用自定义collate_fn
    batch_size = 32
    dataset = ImageFolder(data_source, transform=lambda x: x)  # 保持原始PIL图像
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        collate_fn=collate_fn  # 使用自定义collate函数
    )
    
    # 获取类别映射
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
    print(f"Number of samples: {len(dataset)}")
    print(f"Classes: {idx_to_class}")
    
    # 加载提示词
    with open(prompt_file) as f:
        prompts = json.load(f)['0']
    
    classnames = prompts['classnames']
    templates = prompts['templates']
    n_classes = len(classnames)
    
    # 按照索引顺序准备类别名称
    classnames_text = [classnames[idx_to_class[idx]] for idx in range(n_classes)]
    for class_idx, classname in enumerate(classnames_text):
        print(f'{class_idx}: {classname}')
    
    # 构建零样本分类器
    print("Building zero-shot classifier...")
    zeroshot_weights = zero_shot_classifier(model, processor, classnames_text, templates, device=device)
    print(f"Classifier shape: {zeroshot_weights.shape}")
    
    # 执行零样本评估
    print("Running zero-shot evaluation...")
    results, dump = run_zeroshot(
        model, processor, zeroshot_weights, dataloader, device, 
        metrics=['acc', 'bacc', 'weighted_f1']
    )
    
    # 打印结果
    print("\nZero-shot Classification Results:")
    for k, v in results.items():
        print(f'{k}: {v:.3f}')

In [3]:
model_path = "/data/ckpt/clip-vit-large-patch14-336"
eval(model_path=model_path)

Loading CLIP model from /data/ckpt/clip-vit-large-patch14-336...


/home/hdc/miniconda3/envs/llavanext_llama/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Number of samples: 7180
Classes: {0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}
0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa', 'uninvolved colon mucosa', 'normal colonic mucosa', 'benign epithelium']
7: ['cancer-associated stroma', 'tumor-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
8: ['colorectal adenocarcinoma epithelium', 'colorectal adenocarcinoma', 'tumor', 'adenocarcinoma', 'malignant epithelium']
Building zero-shot classifier...
Classifier shape: torch.Size([7

  0%|          | 0/225 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(tru


Zero-shot Classification Results:
acc: 0.488
bacc: 0.474
weighted_f1: 0.476


In [5]:
# model_path = "/data/ljd/LLaVa/ckpt/stage1_clip/pathclip_ft/results/model/" # 62,58,62
model_path = "/data/ljd/LLaVa/clip/clip-pathology_v1/results/model/" # 69,64,67
eval(model_path=model_path)

Loading CLIP model from /data/ljd/LLaVa/clip/clip-pathology_v1/results/model/...
Number of samples: 7180
Classes: {0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}
0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa', 'uninvolved colon mucosa', 'normal colonic mucosa', 'benign epithelium']
7: ['cancer-associated stroma', 'tumor-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
8: ['colorectal adenocarcinoma epithelium', 'colorectal adenocarcinoma', 'tumor', 'adenocarcinoma', 'malign

  0%|          | 0/225 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(tru


Zero-shot Classification Results:
acc: 0.691
bacc: 0.640
weighted_f1: 0.676


In [2]:
model_path = '/data/ckpt/PathGen-LLaVA/pathgenclip-vit-large-patch14'
eval(model_path=model_path)

Loading CLIP model from /data/ckpt/PathGen-LLaVA/pathgenclip-vit-large-patch14...


/home/hdc/miniconda3/envs/llavanext/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Number of samples: 7180
Classes: {0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}
0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa', 'uninvolved colon mucosa', 'normal colonic mucosa', 'benign epithelium']
7: ['cancer-associated stroma', 'tumor-associated stroma', 'stromal cells', 'stromal tissue', 'stroma']
8: ['colorectal adenocarcinoma epithelium', 'colorectal adenocarcinoma', 'tumor', 'adenocarcinoma', 'malignant epithelium']
Building zero-shot classifier...
Classifier shape: torch.Size([7

  0%|          | 0/225 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(tru


Zero-shot Classification Results:
acc: 0.757
bacc: 0.732
weighted_f1: 0.763


## PathGen-CLIP-L 评估

In [4]:
import os
import torch
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import open_clip
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, 
                            classification_report, roc_auc_score)

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

# 自定义collate函数处理PIL图像
def collate_fn(batch):
    # 将图像和标签分开
    images, targets = zip(*batch)
    # 返回图像列表和标签张量
    return list(images), torch.tensor(targets)

@torch.no_grad()
def zero_shot_classifier(model, tokenizer, preprocess, classnames, templates, device=None):
    """
    构建零样本分类器权重矩阵
    
    Args:
        model: OpenCLIP模型
        tokenizer: OpenCLIP分词器
        preprocess: OpenCLIP图像预处理
        classnames: 每个类别的名称列表的列表（每个类别可能有多个描述）
        templates: 文本提示模板列表
        device: 计算设备
    
    Returns:
        normalized_text_embeddings: 分类器权重矩阵 [embedding_dim, num_classes]
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    zeroshot_weights = []
    
    for classnames_for_class in classnames:
        embeddings_for_class = []
        for classname in classnames_for_class:
            # 使用模板创建提示词
            texts = [template.replace('CLASSNAME', classname) for template in templates]
            
            # 批量处理提示词以提高效率
            text_tokens = tokenizer(texts).to(device)
            
            # 获取文本特征
            with torch.no_grad(), torch.cuda.amp.autocast():
                text_features = model.encode_text(text_tokens)
                text_features = F.normalize(text_features, dim=-1)
            
            # 存储该类名的嵌入
            embeddings_for_class.append(text_features)
        
        # 平均每个类别中所有类名的嵌入
        class_embedding = torch.cat(embeddings_for_class, dim=0)
        class_embedding = class_embedding.mean(dim=0)
        class_embedding = F.normalize(class_embedding, dim=-1)
        
        zeroshot_weights.append(class_embedding)
    
    # 堆叠所有类别的嵌入
    zeroshot_weights = torch.stack(zeroshot_weights, dim=1)
    return zeroshot_weights

@torch.no_grad()
def run_zeroshot(model, preprocess, classifier, dataloader, device, metrics=['bacc', 'weighted_f1']):
    """
    执行零样本分类评估
    
    Args:
        model: OpenCLIP模型
        preprocess: OpenCLIP图像预处理
        classifier: 零样本分类器权重矩阵
        dataloader: 数据加载器
        device: 计算设备
        metrics: 要计算的指标列表
    
    Returns:
        results: 指标结果
        dump: 包含预测和标签的详细信息
    """
    acc_meter = AverageMeter()
    
    logits_all, targets_all, preds_all = [], [], []
    
    for batch_idx, (imgs, targets) in enumerate(tqdm(dataloader)):
        # 处理批量图像
        processed_images = torch.stack([preprocess(img) for img in imgs])
        processed_images = processed_images.to(device)
        
        # 获取图像特征
        with torch.no_grad(), torch.cuda.amp.autocast():
            image_features = model.encode_image(processed_images)
            image_features = F.normalize(image_features, dim=-1)
        
        targets = targets.to(device)
        batch_size = targets.size(0)
        
        # 计算相似度得分
        logits = 100.0 * (image_features @ classifier)
        preds = logits.argmax(dim=1)
        
        # 保存结果
        logits_all.append(logits.cpu().numpy())
        targets_all.append(targets.cpu().numpy())
        preds_all.append(preds.cpu().numpy())
        
        # 更新准确率
        acc_meter.update((preds == targets).float().mean().item(), n=batch_size)
    
    # 合并所有批次的结果
    targets_all = np.concatenate(targets_all, axis=0)
    logits_all = np.concatenate(logits_all, axis=0)
    probs_all = F.softmax(torch.from_numpy(logits_all) / 100.0, dim=1).numpy()
    preds_all = np.concatenate(preds_all, axis=0)
    
    # 计算各种指标
    bacc = balanced_accuracy_score(targets_all, preds_all)
    weighted_kappa = cohen_kappa_score(targets_all, preds_all, weights='quadratic')
    kappa = cohen_kappa_score(targets_all, preds_all)
    cls_rep = classification_report(targets_all, preds_all, output_dict=True, zero_division=0)
    acc = acc_meter.avg
    
    # 计算ROC AUC
    n_classes = probs_all.shape[1]
    if n_classes == 2:
        class_probs = probs_all[:,1]
        roc_kwargs = {}
    else:
        class_probs = probs_all
        roc_kwargs = {'multi_class': 'ovo', 'average': 'macro'}
    
    try:
        roc_auc = roc_auc_score(targets_all, class_probs, **roc_kwargs)
    except ValueError:
        roc_auc = np.nan
    
    # 收集所有计算的指标
    results = {'acc': acc, 
            'bacc': bacc, 
            'weighted_kappa': weighted_kappa,
            'kappa': kappa,
            'roc_auc': roc_auc,
            'weighted_f1': cls_rep['weighted avg']['f1-score']}
    
    # 只保留请求的指标
    results = {k: results[k] for k in metrics if k in results}
    
    # 准备详细输出
    dump = {
        'logits': logits_all,
        'targets': targets_all,
        'preds': preds_all,
    }
    
    return results, dump

def load_model_and_transforms(model_path):
    """加载OpenCLIP模型、分词器和预处理函数"""
    # 对于PathGen-CLIP-L模型
    # 根据您的路径调整模型和架构
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained=model_path)
    tokenizer = open_clip.get_tokenizer('ViT-B-16')
    # model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained=model_path)
    # tokenizer = open_clip.get_tokenizer('ViT-L-14')
    print(f"preprocess: {preprocess}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, tokenizer, preprocess, device

def eval_openclip(model_path=None):
    # 加载模型和处理器
    print(f"Loading OpenCLIP model from {model_path}...")
    model, tokenizer, preprocess, device = load_model_and_transforms(model_path)
    
    # 设置数据路径
    root = Path('.').resolve()
    data_source = '/data/ljd/LLaVa/datasets/crc100k/CRC-VAL-HE-7K/'
    
    # 加载提示词文件
    prompt_file = '/home/hdc/ljd1/github/conch/prompts/crc100k_prompts_all_per_class.json'
    
    # 创建数据加载器 - 使用自定义collate_fn
    batch_size = 32
    dataset = ImageFolder(data_source, transform=lambda x: x)  # 保持原始PIL图像
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4,
        collate_fn=collate_fn  # 使用自定义collate函数
    )
    
    # 获取类别映射
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
    print(f"Number of samples: {len(dataset)}")
    print(f"Classes: {idx_to_class}")
    
    # 加载提示词
    with open(prompt_file) as f:
        prompts = json.load(f)['0']
    
    classnames = prompts['classnames']
    templates = prompts['templates']
    n_classes = len(classnames)
    
    # 按照索引顺序准备类别名称
    classnames_text = [classnames[idx_to_class[idx]] for idx in range(n_classes)]
    for class_idx, classname in enumerate(classnames_text):
        print(f'{class_idx}: {classname}')
    
    # 构建零样本分类器
    print("Building zero-shot classifier...")
    zeroshot_weights = zero_shot_classifier(model, tokenizer, preprocess, classnames_text, templates, device=device)
    print(f"Classifier shape: {zeroshot_weights.shape}")
    
    # 执行零样本评估
    print("Running zero-shot evaluation...")
    results, dump = run_zeroshot(
        model, preprocess, zeroshot_weights, dataloader, device, 
        metrics=['acc', 'bacc', 'weighted_f1']
    )
    
    # 打印结果
    print("\nZero-shot Classification Results:")
    for k, v in results.items():
        print(f'{k}: {v:.3f}')

In [5]:
# 评估PathGen-CLIP-L模型
model_path = "/data/ckpt/PathGen-CLIP-L/pathgen-clip-l.pt"
eval_openclip(model_path=model_path)

Loading OpenCLIP model from /data/ckpt/PathGen-CLIP-L/pathgen-clip-l.pt...
preprocess: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=warn)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x7efbfe140ee0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
Number of samples: 7180
Classes: {0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}
0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa'

100%|██████████| 225/225 [00:30<00:00,  7.27it/s]


Zero-shot Classification Results:
acc: 0.829
bacc: 0.799
weighted_f1: 0.829


In [2]:
model_path = "/data/ckpt/PathGen-CLIP-L/pathgen-clip.pt"
eval_openclip(model_path=model_path)

Loading OpenCLIP model from /data/ckpt/PathGen-CLIP-L/pathgen-clip.pt...
preprocess: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=warn)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x7efbfe140ee0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
Number of samples: 7180
Classes: {0: 'ADI', 1: 'BACK', 2: 'DEB', 3: 'LYM', 4: 'MUC', 5: 'MUS', 6: 'NORM', 7: 'STR', 8: 'TUM'}
0: ['adipose', 'adipose tissue', 'adipocytes', 'fat', 'fat cells']
1: ['background', 'penmarking', 'empty space', 'background artifacts']
2: ['debris', 'colorectal adenocarcinoma debris and necrosis', 'necrosis', 'necrotic debris']
3: ['lymphocytes', 'lymphoid aggregate', 'immune cells', 'lymphoid infiltrate', 'inflammatory cells']
4: ['mucus', 'mucin', 'mucus pool', 'mucin pool']
5: ['smooth muscle', 'smooth muscle tissue', 'muscle', 'muscularis propria', 'muscularis mucosa']
6: ['normal colon mucosa', 

100%|██████████| 225/225 [00:16<00:00, 13.33it/s]



Zero-shot Classification Results:
acc: 0.635
bacc: 0.646
weighted_f1: 0.639
